In [2]:
!pip install boto3 model-registry --quiet

In [3]:
import re


def get_model_name(model_path: str) -> str:
    match = re.match(r"(.+?)-(\d+\.\d+-.*)", model_path)
    if match:
        return match.group(1)
    else:
        return model_path


def get_version_prefix(model_path: str) -> str:
    match = re.match(r"(.+?)-(\d+\.\d+-.*)", model_path)
    if match:
        return match.group(2) + '-'
    else:
        return ""

In [4]:
from dotenv import load_dotenv
import os

local_base_dir = os.path.expanduser("~/shared")
params_file = f"{local_base_dir}/output-2.env"

load_dotenv(params_file)

model_registry_url = os.environ.get('MODEL_REGISTRY_URL')
model_registry_user_token = os.environ.get('OPENSHIFT_API_TOKEN')
model_path = os.environ.get('MODEL_PATH') #"meta-llama/Llama-3.2-1B-Instruct-tuned"

model_registry_port = 443
model_author = "Red Hat"
model_name = get_model_name(model_path) #"meta-llama/Llama"
storage_key = "models"
version_prefix = get_version_prefix(model_path) #"3.2-1B-Instruct-tuned-"

In [5]:
from model_registry import ModelRegistry, utils
import os
registry = ModelRegistry(
    server_address=model_registry_url,
    port=model_registry_port,
    author=model_author,
    user_token=model_registry_user_token
)

/opt/app-root/lib64/python3.12/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "schema" in "DataSet" shadows an attribute in parent "Artifact"
  warnings.warn(


In [6]:
endpoint_url = os.environ.get('AWS_S3_ENDPOINT')
region_name = os.environ.get('AWS_DEFAULT_REGION')
bucket_name = os.environ.get('AWS_S3_BUCKET')

s3_uri = utils.s3_uri_from(
    endpoint=endpoint_url,
    bucket=bucket_name,
    region=region_name,
    path=model_path
)

s3_uri

's3://models/meta-llama/Llama-3.2-1B-Instruct-tuned?endpoint=http://minio-service.utilities.svc.cluster.local:9000&defaultRegion=none'

In [11]:
import re
from model_registry._client import ModelRegistry, StoreError

def get_next_tuned_version_name(
    registry: ModelRegistry,
    model_name: str,
    prefix: str,
):
    try:
        versions = list(registry.get_model_versions(model_name))
    except StoreError as e:
        if "does not exist" in str(e):
            return f"{prefix}1"

    def is_numeric_suffix(vname: str):
        m = re.search(rf"{re.escape(prefix)}(\d+)$", vname)
        if not m:
            return False
        suffix = m.group(1)
        return int(suffix) <= 99999999

    tuned_versions = [
        v for v in versions
        if getattr(v, "name", "").startswith(prefix) and is_numeric_suffix(getattr(v, "name", ""))
    ]

    if not tuned_versions:
        return f"{prefix}1"

    def extract_suffix(vname: str):
        m = re.search(rf"{re.escape(prefix)}(\d+)$", vname)
        return int(m.group(1)) if m else -1

    max_suffix = max(extract_suffix(getattr(v, "name", "")) for v in tuned_versions)
    next_version_num = max_suffix + 1
    return f"{prefix}{next_version_num}"

In [12]:
version = get_next_tuned_version_name(
    registry,
    model_name=model_name,
    prefix=version_prefix
)

model = registry.register_model(
    name=model_name,
    uri=s3_uri,
    version=version,
    model_format_name="",
    model_format_version="",
    storage_key=storage_key
)

In [13]:
version

'3.2-1B-Instruct-tuned-1'

In [14]:
from dotenv import set_key, dotenv_values
from pathlib import Path

params_file = f"{local_base_dir}/output-3.env"
Path(params_file).write_text("")

set_key(params_file, "MODEL_PATH", model_path)
set_key(params_file, "MODEL_NAME", model_name)
set_key(params_file, "MODEL_VERSION", version)

!cat {params_file}

MODEL_PATH='meta-llama/Llama-3.2-1B-Instruct-tuned'
MODEL_NAME='meta-llama/Llama'
MODEL_VERSION='3.2-1B-Instruct-tuned-1'
